In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/pambifree/02-data-audit/__huggingface_repos__.json
/kaggle/input/notebooks/pambifree/02-data-audit/__notebook__.ipynb
/kaggle/input/notebooks/pambifree/02-data-audit/__output__.json
/kaggle/input/notebooks/pambifree/02-data-audit/__results__.html
/kaggle/input/notebooks/pambifree/02-data-audit/custom.css
/kaggle/input/notebooks/pambifree/02-data-audit/gold_test_v1.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/test.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/train.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/val.csv


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import torch, time
print("GPU available:", torch.cuda.is_available(), "| visible devices:", torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

GPU available: True | visible devices: 1
Tesla T4


In [4]:
!find /kaggle/input -name "*.csv" | head -20

/kaggle/input/notebooks/pambifree/02-data-audit/gold_test_v1.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/test.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/train.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/val.csv


In [5]:
import pandas as pd, numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

PUBLIC_DIR = "/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1"   # replace
GOLD_PATH = "/kaggle/input/notebooks/pambifree/02-data-audit/gold_test_v1.csv"  # replace

train = pd.read_csv(f"{PUBLIC_DIR}/train.csv")
val = pd.read_csv(f"{PUBLIC_DIR}/val.csv")
test = pd.read_csv(f"{PUBLIC_DIR}/test.csv")
gold = pd.read_csv(GOLD_PATH)
trainval = pd.concat([train, val], ignore_index=True)

SEED = 42
def cap_groups(df, cap=5, seed=SEED):
    return df.sample(frac=1, random_state=seed).groupby("group").head(cap)

def scores(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    legit_ok = ((y_pred == 0) & (y_true == 0)).sum() / max((y_true == 0).sum(), 1)
    return {"precision": p, "recall": r, "f1": f, "accuracy": accuracy_score(y_true, y_pred),
            "false_alarm_rate": 1 - legit_ok, "balanced_acc": (r + legit_ok) / 2}

test_c = cap_groups(test)
print(trainval.shape, test.shape, gold.shape)


(8639, 5) (2066, 5) (326, 7)


In [6]:
from transformers import AutoTokenizer

tok_names = {"HingBERT": "l3cube-pune/hing-bert",
             "XLM-R base": "xlm-roberta-base",
             "English BERT": "bert-base-uncased"}
toks = {name: AutoTokenizer.from_pretrained(path) for name, path in tok_names.items()}

hv = val[val["language"] == "Hinglish"]
hv = hv.sample(min(150, len(hv)), random_state=1)
for name, tok in toks.items():
    pieces = sum(len(tok.tokenize(t)) for t in hv["text"])
    words = sum(len(t.split()) for t in hv["text"])
    print(f"{name:13} pieces per word on Hinglish text: {pieces / words:.2f}")

example = "Aapka bank account aaj raat block ho jayega. KYC update karein."
for name, tok in toks.items():
    print(f"\n{name}:", tok.tokenize(example))

config.json:   0%|          | 0.00/716 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

HingBERT      pieces per word on Hinglish text: 1.86
XLM-R base    pieces per word on Hinglish text: 1.76
English BERT  pieces per word on Hinglish text: 1.86

HingBERT: ['aa', '##p', '##ka', 'bank', 'account', 'aa', '##j', 'ra', '##at', 'block', 'ho', 'jay', '##ega', '.', 'ky', '##c', 'update', 'ka', '##re', '##in', '.']

XLM-R base: ['▁Aap', 'ka', '▁bank', '▁account', '▁a', 'aj', '▁ra', 'at', '▁block', '▁ho', '▁ja', 'y', 'ega', '.', '▁KY', 'C', '▁update', '▁kare', 'in', '.']

English BERT: ['aa', '##p', '##ka', 'bank', 'account', 'aa', '##j', 'ra', '##at', 'block', 'ho', 'jay', '##ega', '.', 'ky', '##c', 'update', 'ka', '##re', '##in', '.']


In [7]:
from transformers import AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

def embed(tok, model, texts, batch_size=64, max_length=128):
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = tok(texts[i:i + batch_size], padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt").to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                hidden = model(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1)
            pooled = (hidden * mask).sum(1) / mask.sum(1)      # average over the message's tokens
            out.append(pooled.float().cpu())
    return torch.cat(out).numpy()

emb = {}
for label, path in [("HingBERT", "l3cube-pune/hing-bert"), ("XLM-R base", "xlm-roberta-base")]:
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModel.from_pretrained(path).to(device).eval()
    emb[label] = {"trainval": embed(tok, model, list(trainval["text"])),
                  "test": embed(tok, model, list(test["text"])),
                  "gold": embed(tok, model, list(gold["text"]))}
    print(f"{label}: done in {time.time() - t0:.0f} s | vector size {emb[label]['gold'].shape[1]}")
    del model
    torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: l3cube-pune/hing-bert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


HingBERT: done in 20 s | vector size 768


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLM-R base: done in 23 s | vector size 768


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

rows, fitted_emb = [], {}
for label in emb:
    clf = Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=3000))])
    clf.fit(emb[label]["trainval"], trainval["is_scam"])
    fitted_emb[label] = clf
    sets = [("public test (capped)", emb[label]["test"][test_c.index], test_c["is_scam"]),
            ("public test (all rows)", emb[label]["test"], test["is_scam"]),
            ("gold v1", emb[label]["gold"], gold["label"])]
    for set_name, X, y in sets:
        rows.append({"eval set": set_name, "model": f"{label} (frozen) + LogReg",
                     **scores(y, clf.predict(X))})
emb_final = pd.DataFrame(rows).round(3)

BASE_URL = "https://raw.githubusercontent.com/Pambifreeman/hinglish-scam-detector/main/reports/baseline_final_results.csv"
base = pd.read_csv(BASE_URL)
base = base[base["model"].isin(["Majority class", "TF-IDF chars + LogReg"])]
compare = pd.concat([base, emb_final]).set_index(["eval set", "model"]).sort_index()
compare[["precision", "recall", "f1", "false_alarm_rate", "balanced_acc"]]

precision  recall     f1  \
eval set               model                                                    
gold v1                HingBERT (frozen) + LogReg        0.716   0.770  0.742   
                       Majority class                    0.561   1.000  0.719   
                       TF-IDF chars + LogReg             0.688   0.842  0.757   
                       XLM-R base (frozen) + LogReg      0.715   0.699  0.707   
public test (all rows) HingBERT (frozen) + LogReg        0.782   0.960  0.862   
                       Majority class                    0.399   1.000  0.571   
                       TF-IDF chars + LogReg             0.598   0.970  0.740   
                       XLM-R base (frozen) + LogReg      0.783   0.967  0.866   
public test (capped)   HingBERT (frozen) + LogReg        0.951   0.955  0.953   
                       Majority class                    0.528   1.000  0.691   
                       TF-IDF chars + LogReg             0.940   0.966  0.953   
                       XLM-R base (frozen) + LogReg      0.948   0.963  0.955   

                                                     false_alarm_rate  \
eval set               model                                            
gold v1                HingBERT (frozen) + LogReg               0.392   
                       Majority class                           1.000   
                       TF-IDF chars + LogReg                    0.490   
                       XLM-R base (frozen) + LogReg             0.357   
public test (all rows) HingBERT (frozen) + LogReg               0.178   
                       Majority class                           1.000   
                       TF-IDF chars + LogReg                    0.434   
                       XLM-R base (frozen) + LogReg             0.178   
public test (capped)   HingBERT (frozen) + LogReg               0.055   
                       Majority class                           1.000   
                       TF-IDF chars + LogReg                    0.069   
                       XLM-R base (frozen) + LogReg             0.060   

                                                     balanced_acc  
eval set               model                                       
gold v1                HingBERT (frozen) + LogReg           0.689  
                       Majority class                       0.500  
                       TF-IDF chars + LogReg                0.676  
                       XLM-R base (frozen) + LogReg         0.671  
public test (all rows) HingBERT (frozen) + LogReg           0.891  
                       Majority class                       0.500  
                       TF-IDF chars + LogReg                0.768  
                       XLM-R base (frozen) + LogReg         0.895  
public test (capped)   HingBERT (frozen) + LogReg           0.950  
                       Majority class                       0.500  
                       TF-IDF chars + LogReg                0.948  
                       XLM-R base (frozen) + LogReg         0.952

In [9]:
best_label = max(fitted_emb, key=lambda k: scores(gold["label"], fitted_emb[k].predict(emb[k]["gold"]))["balanced_acc"])
print("Best on gold (by balanced accuracy):", best_label)

g = gold.copy()
g["pred"] = fitted_emb[best_label].predict(emb[best_label]["gold"])
g["correct"] = g["pred"] == g["label"]
print(g.groupby("type")["correct"].agg(["mean", "size"]).round(2), "\n")
print(g.groupby("tag")["correct"].agg(["mean", "size"]).round(2).sort_values("mean").head(12))

Best on gold (by balanced accuracy): HingBERT
       mean  size
type             
legit  0.61   143
promo  0.89    65
scam   0.70   118 

                mean  size
tag                       
hinglish_alert  0.20    15
scam_warning    0.30    10
has_link        0.43    21
transactional   0.45    20
obfuscated      0.50    16
very_long       0.50     2
alarming        0.55    20
imitates_legit  0.59    17
shorthand       0.67    24
hinglish_scam   0.68    25
other_scam      0.85    13
urgent_scam     0.86    14


In [10]:
emb_final.to_csv("/kaggle/working/frozen_embedding_results.csv", index=False)